# Disaster Tweet Classification

### College-Level Deep Learning Project

**Goal:** Predict whether a tweet is about a real disaster (`1`) or not (`0`).

Simple flow:

`Tweet → Clean Text → TF-IDF → Model → Prediction → Evaluation`


## 1. Import Libraries

We use pandas for data, scikit-learn for text processing and Logistic Regression, and TensorFlow/Keras for the simple neural network.


In [ ]:
import re
import html
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

## 2. Load the Dataset

`train.csv` contains the tweets and their correct answers. `test.csv` contains tweets for final prediction.

`target = 1` means disaster, and `target = 0` means not disaster.


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training data:", train.shape)
print("Test data:", test.shape)

train.head()


## 3. Clean the Tweets

We make the text easier for the model to understand by:
- converting text to lowercase
- removing links
- removing @mentions
- removing unnecessary symbols


In [ ]:
# Platform filler words that say nothing about disasters
FILLER = r"\b(?:rt|link in bio|swipe up|see more|tag a friend|follow us)\b"

def clean_text(text):
    text = html.unescape(str(text))                                   # &amp; -> &
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)               # links
    text = re.sub(r"@\w+", " ", text)                                 # @mentions
    text = re.sub(r"#(\w+)",                                          # #HashTag -> hash tag
                  lambda m: " " + re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", m.group(1)), text)
    text = text.lower()
    text = re.sub(FILLER, " ", text)                                  # RT, link in bio, ...
    text = re.sub(r"([a-z])\1{2,}", r"\1\1", text)                    # sooooo -> soo
    text = re.sub(r"[^a-z\s]", " ", text)                             # emojis, numbers, symbols
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean_text"] = train["text"].apply(clean_text)
test["clean_text"] = test["text"].apply(clean_text)

print("Original:", train.loc[0, "text"])
print("Cleaned :", train.loc[0, "clean_text"])

# Example of an Instagram-style post
example = "🚨 Wildfire near #LakeTahoe!!! Evacuate NOW &amp; stay safe 🔥🔥 @fireDept link in bio"
print()
print("Original:", example)
print("Cleaned :", clean_text(example))

## 4. Split the Data

We use 80% of the training data to train the model and 20% to evaluate it.


In [ ]:
X = train["clean_text"]
y = train["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))


## 5. Convert Text into Numbers using TF-IDF

Machine learning models cannot directly understand text. TF-IDF converts each tweet into numerical features.

We keep the 5,000 most useful words.


In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test["clean_text"])

print("TF-IDF shape:", X_train_tfidf.shape)


## 6. Model 1: Logistic Regression

This is our simple baseline model. We use it to compare the neural network with a traditional machine learning model.


In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_val_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_val, lr_pred))
print(classification_report(y_val, lr_pred))


## 7. Model 2: Simple Neural Network

The neural network has:
- an input layer containing the TF-IDF features
- one hidden layer with ReLU
- one output neuron with Sigmoid

Sigmoid gives a value between 0 and 1, which is useful for binary classification.


In [ ]:
nn_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_tfidf.shape[1],)),
    Dense(1, activation="sigmoid")
])

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

nn_model.summary()


In [ ]:
history = nn_model.fit(
    X_train_tfidf.toarray(),
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_tfidf.toarray(), y_val),
    verbose=1
)


## 8. Evaluate the Neural Network

In [ ]:
nn_prob = nn_model.predict(X_val_tfidf.toarray(), verbose=0)
nn_pred = (nn_prob >= 0.5).astype(int).ravel()

print("Neural Network Accuracy:", accuracy_score(y_val, nn_pred))
print(classification_report(y_val, nn_pred))


## 9. Compare the Models

We compare the accuracy of the simple machine learning model and the neural network.


In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Neural Network"],
    "Accuracy": [
        accuracy_score(y_val, lr_pred),
        accuracy_score(y_val, nn_pred)
    ]
})

results


## 10. Predict the Test Tweets

The neural network is used to predict whether each test tweet is a disaster tweet.


In [ ]:
test_prob = nn_model.predict(X_test_tfidf.toarray(), verbose=0)
test_pred = (test_prob >= 0.5).astype(int).ravel()

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_pred
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")
submission.head()


## 11. Try It on Instagram and Facebook Posts

The same cleaning, TF-IDF and neural network work on posts from any platform. `predict_posts` cleans the posts, converts them into numbers with the **same TF-IDF** learned earlier, and uses the neural network to predict.

`sample_posts.csv` contains a few example posts written for this demo (they are **not** part of the training data). The last two are deliberately tricky: they use disaster words in a joking way.

In [ ]:
def predict_posts(posts, text_column="text"):
    """Predict disaster (1) or not (0) for posts from any platform."""
    cleaned = posts[text_column].astype(str).apply(clean_text)
    features = tfidf.transform(cleaned).toarray()
    prob = nn_model.predict(features, verbose=0).ravel()

    result = posts.copy()
    result["disaster_probability"] = prob.round(3)
    result["prediction"] = (prob >= 0.5).astype(int)

    # A post with only emojis or links has no words left to judge
    empty = cleaned == ""
    result.loc[empty, "disaster_probability"] = float("nan")
    result["prediction"] = result["prediction"].astype("Int64")
    result.loc[empty, "prediction"] = pd.NA
    return result

sample_posts = pd.DataFrame({
    "platform": ["Instagram", "Facebook", "Instagram", "Facebook",
                 "Instagram", "Facebook", "Instagram", "Facebook"],
    "text": [
        "🚨 Wildfire is spreading fast near our town. Evacuation orders just issued, please leave now #WildfireAlert #Evacuate 🔥",
        "Neighbors, the river has flooded Main Street and several homes are underwater. Rescue crews are on the way. Stay safe everyone.",
        "Magnitude 6.8 earthquake just hit, buildings collapsed downtown. Praying for everyone affected 🙏 #Earthquake",
        "Tornado warning issued for our county, take shelter in your basement immediately.",
        "Golden hour at the beach with my favorite people ☀️🌊 #SunsetVibes #WeekendMood",
        "Excited to announce our bakery's new sourdough menu! Come by this Saturday for a free tasting.",
        "This new playlist is absolute fire 🔥🔥🔥 link in bio #NowPlaying",
        "My presentation today was a total disaster lol, I forgot every slide 😂",
    ],
})
predict_posts(sample_posts)

## 12. Use Your Own Posts

1. Save your Instagram or Facebook captions in a CSV file with a column named `text` (a `platform` column is optional).
2. Remove the `#` from the three lines below and run the cell.

The model learned from tweets, so treat its answers on other platforms as a good starting point, not a final verdict. Captions that are longer or use more slang can be harder. To improve it, label some of your own posts and add them to `train.csv` using the same `id`, `text`, `target` columns.

In [ ]:
while True:
    post = input("Paste an Instagram / Facebook / Twitter post (or type 'quit'): ").strip()
    if post.lower() == "quit":
        print("Done.")
        break
    if post == "":
        continue

    result = predict_posts(pd.DataFrame({"text": [post]})).iloc[0]

    if pd.isna(result["prediction"]):
        print("-> No words left to judge (only emojis or links).\n")
    else:
        label = "DISASTER" if result["prediction"] == 1 else "NOT a disaster"
        print(f"-> {label} (disaster probability: {result['disaster_probability']:.1%})\n")

## Conclusion

The project classifies social media posts (Twitter, Instagram, Facebook) as disaster or non-disaster posts.

TF-IDF converts text into numbers, Logistic Regression provides a simple baseline, and the neural network learns patterns from the numerical features.

### Main concepts demonstrated
- Text preprocessing (including hashtags and emojis)
- TF-IDF
- Binary classification
- Neural network
- ReLU activation
- Sigmoid activation
- Model training
- Model evaluation
- Final prediction
- Applying one model to several social media platforms